# Lab 6.3 - Model Context Protocol (MCP): Connecting LLMs to External Services
**Module 6: Agentic AI & Tool Use**

In this lab you will:
- Understand the **MCP (Model Context Protocol)** open standard and how it connects LLMs to tools
- Learn the **three MCP primitives**: Tools, Resources, and Prompts
- Define **MCP tool schemas** using JSON Schema format
- Implement a full **agentic loop** that autonomously calls tools to resolve an incident

> **Instructor Note:** MCP turns a chat assistant into an agent. Before MCP, every team wrote custom glue code to connect their LLM to their tools. MCP standardises that contract so any MCP-compatible LLM can talk to any MCP-compatible service without bespoke integration work. The agentic loop at the end of this lab is the payoff — watch Gemini decide autonomously which tools to call and in what order: GitHub first to search for known issues, then create a new GitHub issue to track the incident if none already exists.

## 📦 Requirements & Troubleshooting

### ✅ Before You Start — Checklist

Work through this top-to-bottom before running any code cell:

**Step 1 — Python version**
Open a terminal and run:
```bash
python3 --version
```
You need **Python 3.10, 3.11, 3.12, or 3.13**. Python 3.14 is not supported by some packages.
In VS Code, check the kernel shown top-right. If it shows 3.14, click it → *Select Another Kernel* → pick 3.11 or 3.12.

**Step 2 — Virtual environment (recommended) or If facing issue try using global env**
```bash
cd "AI:ML intermediate"
source myenv/bin/activate          # activate the workshop venv
```
If `myenv` doesn't exist yet:
```bash
python3.11 -m venv myenv && source myenv/bin/activate
```

**Step 3 — Install packages**
```bash
pip install google-generativeai pandas
```

**Step 4 — Set your Gemini API key**
Create a file called `.env` in the `AI:ML intermediate/` folder (the project root):
```
GEMINI_API_KEY=AIza...your-key-here...
```
Get a free key at: https://aistudio.google.com/app/apikey

> Keys starting with `AQ.` also work — they are a newer Google credential format.

**Step 5 — Confirm kernel = your venv**
In VS Code, the kernel shown top-right must match your activated environment.
If it shows `Python 3.x (myenv)` you are good. If it shows a system Python, click it and select the myenv kernel.

---

### 📦 Required Packages

| Package | Install command |
|---------|----------------|
| google-generativeai | `pip install google-generativeai` |
| pandas | `pip install pandas` |
| requests | `pip install requests` |

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named 'google.generativeai'`**
> Package missing from this kernel's Python.
> Fix: In a terminal with the venv active, run `pip install google-generativeai`, then **Kernel → Restart** in VS Code.

**`CalledProcessError` — exit status 1 or `--break-system-packages`**
> The auto-install cell failed because pip is protected.
> Fix: Open a terminal, run `source myenv/bin/activate`, then `pip install google-generativeai pandas`.

**`404 models/gemini-1.5-flash is not found`**
> `gemini-1.5-flash` is **deprecated** and removed from the API.
> Fix: The setup cell (Cell 4) automatically picks `gemini-3.1-flash-lite`. If you hardcoded `gemini-1.5-flash` anywhere, replace it with `gemini-3.1-flash-lite`.

**`ResourceExhausted: 429 Quota exceeded — limit: 0`**
> Your API key has no quota for that specific model.
> Fix: The setup cell tries multiple models automatically. If all fail, wait 60 seconds and re-run Cell 4. The free tier resets per minute.

**`InvalidArgument: API key not valid`**
> Stale or malformed key.
> Fix: Regenerate at aistudio.google.com, update `.env`, then restart the kernel.

**`AttributeError: module 'google.generativeai' has no attribute 'protos'`**
> Old version of the library installed.
> Fix: `pip install --upgrade google-generativeai`, then restart the kernel.

**Gemini prints a FutureWarning about `google.generativeai`**
> This is a deprecation warning only — the library still works fine for this lab.
> You can safely ignore it. The `warnings.filterwarnings('ignore')` line in Cell 4 suppresses it.

---

> **Note — Mock vs Live APIs:**
> This lab uses a **mock GitHub** class. No real account or token is needed for the notebook cells.
> The agentic loop, tool schemas, and function-calling patterns are identical to what you would use with a real GitHub MCP server. To go live, swap the mock service calls for real HTTP requests — the LLM-facing code does not change.
> Section 2 shows how to connect VS Code to the **real** GitHub MCP server using your GitHub token.

In [12]:
import subprocess, sys, warnings
warnings.filterwarnings('ignore')

required = {
    'google.generativeai': 'google-generativeai',
    'pandas':              'pandas',
    'requests':            'requests',
}

for pkg, inst in required.items():
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {inst}...')
        # Try without --break-system-packages first (works inside a venv)
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', inst, '--quiet'],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            # Fall back for system Python environments
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', inst,
                 '--quiet', '--break-system-packages']
            )
        print(f'  {inst} installed.')

# Verify imports work
try:
    import google.generativeai, pandas, requests
    print('All packages ready ✅')
except ImportError as e:
    print(f'❌ Import failed after install: {e}')
    print('Fix: open a terminal, activate your venv, run:')
    print('  pip install google-generativeai pandas requests')
    print('Then: Kernel → Restart Kernel in VS Code.')


All packages ready ✅


In [13]:
import os, json, time, requests, pathlib, warnings
warnings.filterwarnings('ignore')
import google.generativeai as genai

# ── Load API key from .env file or environment variable ──────────────────
# Priority: environment variable → .env in current folder → .env two levels up
_env_path = pathlib.Path('.env')
if not _env_path.exists():
    _env_path = pathlib.Path('../../.env')
if _env_path.exists():
    for _line in _env_path.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith('#') and '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k.strip(), _v.strip())

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

if not GEMINI_API_KEY:
    # ── Paste your key here if .env is not working ────────────────────────
    # os.environ['GEMINI_API_KEY'] = 'AIza...'   # ← uncomment and paste key
    raise EnvironmentError(
        'GEMINI_API_KEY not set.\n'
        'Option 1 (recommended): create a .env file in the AI:ML intermediate/ folder:\n'
        '  GEMINI_API_KEY=AIza...your-key...\n'
        'Option 2: uncomment the line above this error and paste your key directly.\n'
        'Get a free key at: https://aistudio.google.com/app/apikey'
    )

genai.configure(api_key=GEMINI_API_KEY)

# ── Auto-select a working Gemini model ───────────────────────────────────
# gemini-1.5-flash is deprecated (removed from API). gemini-3.1-flash-lite
# is the recommended free-tier model as of May 2026. The function below
# tries each candidate in order and uses the first one that responds.
def _find_working_model(candidates):
    for name in candidates:
        try:
            inst = genai.GenerativeModel(model_name=name)
            inst.generate_content(
                'ping',
                generation_config=genai.GenerationConfig(max_output_tokens=2)
            )
            return name
        except Exception:
            continue
    raise RuntimeError(
        'No working Gemini model found for this API key.\n'
        'Check: (1) key is valid, (2) not quota-exhausted, (3) billing enabled for paid models.\n'
        'Wait 60 seconds and re-run this cell — free-tier quota resets per minute.'
    )

MODEL = _find_working_model([
    'gemini-3.1-flash-lite',   # recommended: fast, free-tier compatible
    'gemini-flash-lite-latest', # alias fallback
    'gemini-2.5-flash',        # if free-tier flash-lite is exhausted
    'gemini-2.0-flash',        # paid-tier fallback
])
print(f'Using model: {MODEL}')

# ── Reusable single-turn helper ───────────────────────────────────────────
def call_gemini(prompt: str, system: str = '', model: str = None,
                temperature: float = 0.2) -> str:
    """Send a single prompt to Gemini and return the response text."""
    cfg = genai.GenerationConfig(temperature=temperature)
    m = genai.GenerativeModel(
        model_name=model or MODEL,
        system_instruction=system or None,
        generation_config=cfg,
    )
    return m.generate_content(prompt).text.strip()

# ── Smoke test ────────────────────────────────────────────────────────────
reply = call_gemini('Reply with exactly three words: MCP lab ready')
print(f'Gemini says: {reply}')
print('\nAPI setup complete ✅  — all remaining cells will use MODEL =', MODEL)


Using model: gemini-3.1-flash-lite
Gemini says: MCP lab ready.

API setup complete ✅  — all remaining cells will use MODEL = gemini-3.1-flash-lite


## Section 1 - What is MCP (Model Context Protocol)?

**Model Context Protocol (MCP)** is an open protocol published by Anthropic in November 2024. It standardises the way LLM-powered applications connect to external data sources and tools — the same way HTTP standardised how browsers talk to web servers.

Before MCP, every team wrote custom glue code: a bespoke GitHub integration, a bespoke database connector, each one different. MCP defines a single contract so that any MCP client (Cursor, Claude Desktop, your own app) can talk to any MCP server (GitHub, PostgreSQL, internal APIs) without custom work.

### The Three MCP Primitives

| Primitive | What it is | Example |
|-----------|-----------|--------|
| **Tools** | Functions the LLM can call (read + write) | `create_github_issue`, `search_github_issues` |
| **Resources** | Read-only data the LLM can access | Open incidents list, runbook KB articles |
| **Prompts** | Reusable prompt templates with arguments | `incident_summary_prompt(severity, component)` |

### Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                        MCP Architecture                         │
│                                                                  │
│  ┌──────────────────┐    JSON-RPC 2.0     ┌───────────────────┐ │
│  │   MCP Client     │◄──────────────────►│  GitHub MCP       │ │
│  │  (Cursor / your  │                     │  Server           │ │
│  │   Python app)    │    JSON-RPC 2.0     ├───────────────────┤ │
│  │                  │◄──────────────────►│  PostgreSQL MCP   │ │
│  │  ┌────────────┐  │                     │  Server           │ │
│  │  │  Gemini /  │  │    JSON-RPC 2.0     ├───────────────────┤ │
│  │  │  Claude /  │  │◄──────────────────►│  Internal API     │ │
│  │  │  GPT-4     │  │                     │  MCP Server       │ │
│  │  └────────────┘  │                     └───────────────────┘ │
│  └──────────────────┘                                            │
│                                                                  │
│  Transport: stdio (local) or HTTP+SSE (remote)                  │
└─────────────────────────────────────────────────────────────────┘
```

### How a tool call works

1. Client sends the LLM a list of available tool schemas (JSON Schema)
2. LLM responds with a structured `tool_call` — name + arguments
3. Client executes the tool against the MCP server
4. Client sends the tool result back to the LLM
5. LLM incorporates the result and decides whether to call more tools or respond

> **Instructor Note:** MCP servers are typically local processes communicating over stdio — not remote APIs. The GitHub MCP server, for example, runs as `npx @modelcontextprotocol/server-github` on the developer's machine and proxies calls to the GitHub REST API using the developer's token. This means credentials stay local and never go to the LLM provider.

## Section 2 - Step-by-Step: Adding GitHub MCP to VS Code & Cursor

This section is a complete practical guide. Follow each step in order.

---

### Step 1 — Install Node.js (prerequisite)

MCP servers run as local Node.js processes. Check if you already have it:

```bash
node --version   # must be v18 or higher
npm --version
```

If not installed:
- **macOS:** `brew install node`
- **Windows:** download from https://nodejs.org (LTS version)
- **Linux:** `sudo apt install nodejs npm`

Verify after install:
```bash
node --version   # should print v18.x or higher
npx --version    # should print 10.x or higher
```

> **Why npx?** MCP servers are npm packages. `npx` downloads and runs them without a global install — each server runs in an isolated process on your machine.

---

### Step 2 — Get a GitHub Personal Access Token

1. Go to https://github.com/settings/tokens?type=beta
2. Click **Generate new token (fine-grained)**
3. Set **Expiration** → 90 days
4. Under **Permissions**, enable: `Issues: Read and write`, `Pull requests: Read`
5. Click **Generate token** → copy the token (starts with `github_pat_`)

> Keep this token safe — you only see it once.

---

### Step 3A — Configure GitHub MCP in Cursor

**Option A: Cursor Settings UI (easiest)**

1. Open Cursor
2. Press `Cmd+,` (macOS) or `Ctrl+,` (Windows) → search `MCP`
3. Click **Edit in settings.json** under "Cursor: MCP Servers"
4. Add the configuration below (fill in your token):

```json
{
  "cursor.mcp.servers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_PERSONAL_ACCESS_TOKEN": "github_pat_YOUR_TOKEN_HERE"
      }
    }
  }
}
```

5. Save → **fully quit and reopen Cursor** (not just reload window)

**Option B: Cursor MCP panel**

1. Cursor → **⚙ gear icon** bottom-left → **Cursor Settings** → **Features** → **MCP**
2. Click **+ Add new MCP server**
3. Name = `github`, Command = `npx`, Args = `-y @modelcontextprotocol/server-github`
4. Add env var: `GITHUB_PERSONAL_ACCESS_TOKEN` = your token → **Save**

---

### Step 3B — Configure GitHub MCP in VS Code

VS Code 1.99+ supports MCP natively through Copilot Chat.

**Create `.vscode/mcp.json` in your project root:**

```bash
mkdir -p ".vscode"
```

Paste this content (fill in your token):

```json
{
  "servers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_PERSONAL_ACCESS_TOKEN": "github_pat_YOUR_TOKEN"
      }
    }
  }
}
```

VS Code detects `mcp.json` and starts the server automatically when you open Copilot Chat.

---

### Step 4 — Verify GitHub MCP is working

**In Cursor:**
1. Open AI chat (`Cmd+L`)
2. Type: `What GitHub issues are open right now?`
3. Cursor shows a **tool call indicator** — then lists your real GitHub issues

**In VS Code:**
1. Open Copilot Chat (`Ctrl+Alt+I` / `Cmd+Shift+I`)
2. Type: `List my open GitHub issues`
3. VS Code calls the GitHub MCP server and returns results with a tool-call badge

**Quick test (no real issues needed):**
```
List the tools available to you right now
```
You will see `search_issues`, `create_issue`, `get_pull_request`, etc. in the response.

---

### Step 5 — Nutanix-specific GitHub MCP prompts

Once connected, try these in Cursor or VS Code Copilot Chat:

```
Search open GitHub issues containing "stargate crash"
```

```
Create a GitHub issue titled "Stargate crash loop on node-3 - AOS 6.7.2"
with label "P1" and body: "Stargate entered crash loop after NVMe disk failure.
Workaround: restart stargate service."
```

```
Search issues with label "disk-io" in state open
```

---

### Common GitHub MCP Setup Errors

**`Cannot find module '@modelcontextprotocol/server-github'`**
> npx failed to download the package.
> Fix: check internet, then run: `npx -y @modelcontextprotocol/server-github`

**`Error: GITHUB_PERSONAL_ACCESS_TOKEN is not set`**
> Token missing from the env block.
> Fix: confirm the token is in the `env` section (not `args`), with no extra spaces.

**`Failed to connect to MCP server`** in Cursor
> Server crashed on startup.
> Fix: open Cursor Output panel (`View → Output` → select `MCP`) to see the error.
> Most common cause: Node.js not on PATH. Run `which node` in terminal to confirm.

**MCP tools not appearing in chat**
> Editor did not reload after config change.
> Fix: fully quit and reopen. Check the MCP panel shows green ✅ next to `github`.

**`401 Unauthorized` when calling GitHub tools**
> Token wrong, expired, or missing permissions.
> Fix: regenerate at https://github.com/settings/tokens, update the config, restart editor.

---

> **Instructor Note:** For the workshop, students do NOT need to set up a real GitHub MCP server — the mock `MockGitHub` class in the following cells demonstrates the exact same patterns. The guide above is for students who want to connect a real GitHub repo after the workshop. The key insight: the Python code in Sections 4–9 of this lab is what runs *inside* an MCP server. Understanding it means you can write your own MCP server for any internal Nutanix API.

## Section 3 - MCP Tool Schemas

Every MCP tool is described by a **JSON Schema** definition. This schema is sent to the LLM so it knows what tools are available, what arguments each tool takes, and what each argument means.

The schema has three top-level fields:
- `name` — machine-readable identifier (snake_case)
- `description` — natural language explanation **for the LLM** (this is prompt engineering)
- `inputSchema` — JSON Schema object defining all parameters

### Tool 1: `search_github_issues`

```json
{
  "name": "search_github_issues",
  "description": "Search GitHub issues for known bugs or existing work related to a problem. ALWAYS call this first to check if an issue already exists before creating a new one.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "query": { "type": "string", "description": "Search keywords, e.g. 'stargate disk IO crash loop'" },
      "state": { "type": "string", "enum": ["open", "closed", "all"],
                 "description": "Filter by issue state. Default: open" }
    },
    "required": ["query"]
  }
}
```

### Tool 2: `create_github_issue`

```json
{
  "name": "create_github_issue",
  "description": "Create a new GitHub issue to track an incident or bug. Call this AFTER searching — only if no existing open issue covers the same problem. Returns the new issue number and URL.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "title":  { "type": "string", "description": "One-line issue title" },
      "body":   { "type": "string", "description": "Detailed description including root cause, impact, causal chain, and remediation steps" },
      "labels": { "type": "array", "items": { "type": "string" },
                  "description": "Labels such as ['P1', 'stargate', 'disk-failure', 'aiops']" }
    },
    "required": ["title", "body"]
  }
}
```

> **Instructor Note:** Notice that `description` fields are written **for the LLM**, not for humans. Phrases like "ALWAYS call this first" and "only if no existing open issue covers the same problem" are prompt engineering embedded in the schema. The quality of these descriptions directly controls agent behaviour.

In [14]:
# ── Section 4: Mock GitHub Service ───────────────────────────────────────
# MockGitHub simulates the GitHub Issues API so no real account or token
# is needed. The LLM sees identical tool schemas; only step 3 of the
# agentic loop changes when you go to production (swap the mock call for
# a real requests.get / requests.post to api.github.com).

import json
from datetime import datetime


class MockGitHub:
    """In-memory mock of the GitHub Issues API (pre-seeded with Nutanix issues)."""

    def __init__(self):
        self.issues = [
            {
                'number':    1042,
                'title':     'Stargate crash loop triggered by sustained disk IO errors on NVMe',
                'state':     'open',
                'priority':  'P1',
                'labels':    ['bug', 'stargate', 'disk-io', 'P1'],
                'body':      'Stargate enters crash loop when disk_manager reports sustained IO errors. '
                             'Seen on AOS 6.7.x with NVMe drives. Workaround: restart stargate service.',
                'url':       'https://github.com/nutanix/aos/issues/1042',
                'created_at':'2024-11-15T10:23:00Z',
            },
            {
                'number':    1038,
                'title':     'CVM memory OOM kills Cassandra under high metadata write load',
                'state':     'open',
                'priority':  'P2',
                'labels':    ['bug', 'cvm', 'cassandra', 'memory', 'P2'],
                'body':      'CVM heap exhaustion causes Cassandra OOM during metadata-heavy workloads. '
                             'Increase CVM memory reservation as workaround.',
                'url':       'https://github.com/nutanix/aos/issues/1038',
                'created_at':'2024-11-10T08:45:00Z',
            },
            {
                'number':    1031,
                'title':     'Cerebro replication lag exceeds RPO during peak backup windows',
                'state':     'closed',
                'priority':  'P2',
                'labels':    ['bug', 'cerebro', 'replication', 'P2', 'fixed-6.8'],
                'body':      'Cerebro scheduler starvation during concurrent snapshot + backup. '
                             'Fixed in AOS 6.8.0 — upgrade recommended.',
                'url':       'https://github.com/nutanix/aos/issues/1031',
                'created_at':'2024-10-28T14:00:00Z',
            },
        ]
        self._issue_counter = max(i['number'] for i in self.issues) + 1

    def search_issues(self, query: str, state: str = 'open') -> list:
        """Return issues whose title or body contain any keyword from query."""
        keywords = query.lower().split()
        results = []
        for issue in self.issues:
            if state != 'all' and issue['state'] != state:
                continue
            text = (issue['title'] + ' ' + issue['body']).lower()
            if any(kw in text for kw in keywords):
                results.append(issue)
        return results

    def create_issue(self, title: str, body: str, labels: list = None) -> dict:
        """Create a new issue and append it to the in-memory store."""
        number = self._issue_counter
        self._issue_counter += 1
        issue = {
            'number':    number,
            'title':     title,
            'state':     'open',
            'priority':  'P1' if 'P1' in (labels or []) else 'P2',
            'labels':    labels or [],
            'body':      body,
            'url':       f'https://github.com/nutanix/aos/issues/{number}',
            'created_at': datetime.utcnow().isoformat() + 'Z',
        }
        self.issues.append(issue)
        return {'number': number, 'url': issue['url'], 'state': 'open'}


# ── Instantiate mock service ──────────────────────────────────────────────
github = MockGitHub()

# ── Smoke tests ───────────────────────────────────────────────────────────
print('=== MockGitHub search smoke test ===')
results = github.search_issues('stargate crash')
for i in results:
    print(f"  #{i['number']} [{i['state']}] {i['title']}")

print()
print('=== MockGitHub create smoke test ===')
new_issue = github.create_issue(
    title='Test issue — smoke test',
    body='Testing create_issue method.',
    labels=['test'],
)
print(json.dumps(new_issue, indent=2))

# Clean up the test issue
github.issues = [i for i in github.issues if i['labels'] != ['test']]
github._issue_counter -= 1
print()
print(f'Mock GitHub ready. Pre-seeded issues: {len(github.issues)}')
print('Mock service ready.')

=== MockGitHub search smoke test ===
  #1042 [open] Stargate crash loop triggered by sustained disk IO errors on NVMe

=== MockGitHub create smoke test ===
{
  "number": 1043,
  "url": "https://github.com/nutanix/aos/issues/1043",
  "state": "open"
}

Mock GitHub ready. Pre-seeded issues: 3
Mock service ready.


## Section 5 - MCP Tool Definitions for Gemini

Gemini's **function calling** feature is the Python-side equivalent of MCP. Instead of an MCP server sending tool schemas over JSON-RPC, we define them directly as `FunctionDeclaration` objects using `genai.protos`.

The flow is identical to MCP:
1. We pass tool schemas to Gemini alongside the prompt
2. Gemini returns a `FunctionCall` part (name + args) instead of text
3. We execute the function locally (against our mock services)
4. We send a `FunctionResponse` back to Gemini
5. Gemini incorporates the result and continues

In a production system, step 3 would call the real MCP server. The LLM never changes.

In [15]:
# ── Section 5: Gemini FunctionDeclaration tool definitions ───────────────

import google.generativeai as genai
from google.generativeai import protos

# Tool 1 — search_github_issues
SEARCH_GITHUB_ISSUES = protos.FunctionDeclaration(
    name='search_github_issues',
    description=(
        'Search GitHub issues for known bugs or existing work related to a problem. '
        'ALWAYS call this first before creating a new issue, to check if the '
        'problem is already tracked in the engineering backlog.'
    ),
    parameters=protos.Schema(
        type=protos.Type.OBJECT,
        properties={
            'query': protos.Schema(type=protos.Type.STRING,
                                   description='Search keywords, e.g. stargate disk IO crash loop'),
            'state': protos.Schema(type=protos.Type.STRING,
                                   description='Filter by issue state: open, closed, or all. Default: open'),
        },
        required=['query'],
    ),
)

# Tool 2 — create_github_issue
CREATE_GITHUB_ISSUE = protos.FunctionDeclaration(
    name='create_github_issue',
    description=(
        'Create a new GitHub issue to track an incident or bug. '
        'Call this AFTER searching — only if no existing open issue covers the same problem. '
        'Returns the new issue number and URL.'
    ),
    parameters=protos.Schema(
        type=protos.Type.OBJECT,
        properties={
            'title':  protos.Schema(type=protos.Type.STRING,
                                    description='One-line issue title'),
            'body':   protos.Schema(type=protos.Type.STRING,
                                    description='Detailed description: root cause, impact, causal chain, remediation steps'),
            'labels': protos.Schema(
                type=protos.Type.ARRAY,
                items=protos.Schema(type=protos.Type.STRING),
                description='Labels such as [P1, stargate, disk-failure, aiops]'
            ),
        },
        required=['title', 'body'],
    ),
)

# Wrap both tools in a Tool object
tools = protos.Tool(function_declarations=[
    SEARCH_GITHUB_ISSUES,
    CREATE_GITHUB_ISSUE,
])

tool_names = [fd.name for fd in tools.function_declarations]
print('Registered tools:')
for name in tool_names:
    print(f'  - {name}')

Registered tools:
  - search_github_issues
  - create_github_issue


In [16]:
# ── Section 6: Tool Execution Engine ─────────────────────────────────────
# ToolExecutor maps function names to MockGitHub calls.
# In production, replace mock calls with real GitHub API requests.

class ToolExecutor:
    """Executes tool calls by routing to the appropriate mock service."""

    def __init__(self, github: MockGitHub):
        self.github   = github
        self.call_log = []

    def execute(self, function_name: str, args: dict) -> str:
        """Execute a tool call and return the result as a JSON string."""
        self.call_log.append({'tool': function_name, 'args': args})

        if function_name == 'search_github_issues':
            issues = self.github.search_issues(
                query = args['query'],
                state = args.get('state', 'open'),
            )
            result = {
                'total_count': len(issues),
                'issues': [
                    {'number': i['number'], 'title': i['title'],
                     'state': i['state'], 'priority': i['priority'],
                     'url': i['url'], 'body': i['body'][:200]}
                    for i in issues
                ],
            }
            return json.dumps(result)

        elif function_name == 'create_github_issue':
            result = self.github.create_issue(
                title  = args['title'],
                body   = args['body'],
                labels = args.get('labels', []),
            )
            return json.dumps(result)

        else:
            return json.dumps({'error': f'Unknown tool: {function_name}'})


# ── Instantiate and test ──────────────────────────────────────────────────
executor = ToolExecutor(github)

print('=== Test: search_github_issues ===')
r1 = executor.execute('search_github_issues', {'query': 'stargate crash disk', 'state': 'open'})
print(json.dumps(json.loads(r1), indent=2))

print('\n=== Test: create_github_issue ===')
r2 = executor.execute('create_github_issue', {
    'title':  'Stargate crash loop — node-3 disk IO errors',
    'body':   'Stargate entering crash loop due to sustained disk IO errors on sda3.',
    'labels': ['P1', 'stargate', 'disk-failure', 'aiops'],
})
print(json.dumps(json.loads(r2), indent=2))

# Reset so agentic loop starts fresh
github.issues = github.issues[:3]
github._issue_counter = 1044
executor.call_log.clear()

print('\nTool executor ready.')

=== Test: search_github_issues ===
{
  "total_count": 1,
  "issues": [
    {
      "number": 1042,
      "title": "Stargate crash loop triggered by sustained disk IO errors on NVMe",
      "state": "open",
      "priority": "P1",
      "url": "https://github.com/nutanix/aos/issues/1042",
      "body": "Stargate enters crash loop when disk_manager reports sustained IO errors. Seen on AOS 6.7.x with NVMe drives. Workaround: restart stargate service."
    }
  ]
}

=== Test: create_github_issue ===
{
  "number": 1043,
  "url": "https://github.com/nutanix/aos/issues/1043",
  "state": "open"
}

Tool executor ready.


## Section 7 - The Agentic Loop

The **agentic loop** is the core pattern that separates a chatbot from an agent. Instead of a single prompt → response, the loop allows the LLM to call tools repeatedly until it has gathered enough information to produce a final answer.

### 4-Step Loop

1. **User provides incident** — the agent receives a structured incident summary
2. **LLM calls tools** — Gemini returns a `FunctionCall` part (not text) naming the tool and its arguments
3. **Execute and return** — our executor runs the tool against the mock service and sends a `FunctionResponse` back
4. **Repeat until final text** — the loop continues until Gemini returns a text response with no more tool calls

### Flow for a Disk Failure Incident

```
Disk failure log analysis
         │
         ▼
  Gemini (with tools)
         │
         ├──[Tool call 1]──► search_github_issues("stargate disk IO")
         │                          │
         │                          ▼
         │                  #1042 open P1 found
         │                          │
         │◄─────────────────────────┘
         │
         │  (existing issue found — skip create)
         │        OR
         ├──[Tool call 2]──► create_github_issue(title, body, labels)
         │   (if no existing      │
         │    open issue found)   ▼
         │                  #1044 created
         │                          │
         │◄─────────────────────────┘
         │
         ▼
  Final response (2-3 sentence summary)
```

> **Instructor Note:** The key insight is that the LLM makes decisions based on intermediate results. If `search_github_issues` returns an existing open P1 issue, Gemini will note it in the summary and skip creating a duplicate. If search returns zero results, it creates a new issue. The LLM is reasoning across tool calls — not just executing a fixed script.

In [17]:
# ── Section 8: Agentic Loop Implementation ───────────────────────────────

AGENT_SYSTEM = """You are a senior Nutanix AIOps incident response agent.

When given an incident summary, follow this EXACT sequence:
1. Call search_github_issues — search for known bugs related to the root cause
   and affected component. Use keywords from the incident title and component name.
2. If the search returns an EXISTING OPEN issue that matches, do NOT create a duplicate.
   Skip to the final summary and reference the existing issue number and URL.
3. If the search returns NO matching open issues, call create_github_issue to
   create a new issue. Include the full incident context in the body:
   root cause, causal chain, severity, and remediation steps.

After all tool calls are complete, provide a concise 2-3 sentence summary:
what was found (or created) in GitHub, and what the operator should do next."""


def run_agentic_loop(incident_summary: str, max_turns: int = 10) -> dict:
    """
    Run the agentic loop for an incident.

    Args:
        incident_summary: Structured incident description string.
        max_turns: Safety limit on tool-call iterations.

    Returns:
        dict with keys: final_response, tool_calls, turns
    """
    model = genai.GenerativeModel(
        model_name=MODEL,
        tools=[tools],
        system_instruction=AGENT_SYSTEM,
        generation_config=genai.GenerationConfig(temperature=0.1),
    )

    chat = model.start_chat()
    response = chat.send_message(incident_summary)

    tool_calls_made = []
    final_text = ''
    turns = 0

    # Agentic loop — continue until no more function calls
    while turns < max_turns:
        turns += 1
        has_function_call = False
        function_responses = []

        for part in response.parts:
            if hasattr(part, 'function_call') and part.function_call.name:
                has_function_call = True
                fc = part.function_call
                tool_name = fc.name
                tool_args = dict(fc.args)

                print(f'  [Turn {turns}] Calling tool: {tool_name}')
                print(f'           Args: {list(tool_args.keys())}')

                result_str  = executor.execute(tool_name, tool_args)
                result_data = json.loads(result_str)
                tool_calls_made.append({'tool': tool_name, 'args': tool_args,
                                        'result': result_data})

                print(f'           Result: {result_str[:120]}')

                function_responses.append(
                    protos.Part(
                        function_response=protos.FunctionResponse(
                            name=tool_name,
                            response={'result': result_data},
                        )
                    )
                )

        if not has_function_call:
            for part in response.parts:
                if hasattr(part, 'text') and part.text:
                    final_text += part.text
            break

        response = chat.send_message(function_responses)

    return {
        'final_response': final_text,
        'tool_calls':     tool_calls_made,
        'turns':          turns,
    }


print('run_agentic_loop() defined')

run_agentic_loop() defined


In [18]:
# ── Section 9: Run the Agentic Loop ──────────────────────────────────────
# Scenario A: Cerebro replication incident — NOT in the pre-seeded open issues
# (the only Cerebro issue is CLOSED). Agent should search → then create a new issue.

cerebro_incident = {
    'incident_title':     'Cerebro replication lag causing RPO violations during backup window',
    'root_cause':         'Cerebro scheduler starvation during concurrent snapshot + backup jobs. '
                          'Replication lag exceeded 4-hour RPO threshold on 3 VMs.',
    'affected_component': 'cerebro / replication',
    'severity':           'P2',
    'confidence':         0.91,
    'causal_chain':       [
        'Backup window started at 02:00 — 12 concurrent snapshot jobs',
        'Cerebro replication threads starved by snapshot I/O priority',
        'Replication lag grew from 8 min to 4h22m over 90 minutes',
        'RPO alert fired at 04:22 — 3 protected VMs exceeded 4h threshold',
    ],
    'remediation_steps':  [
        'ncli protection-domain list-replication-status  # check all PD lag',
        'Pause non-critical backup jobs during peak replication window',
        'Increase cerebro thread priority: edit /etc/nutanix/cerebro.gflags',
        'Consider scheduling backups outside replication windows',
    ],
}

incident_text = f"""Incident Title: {cerebro_incident['incident_title']}
Severity: {cerebro_incident['severity']}
Confidence: {cerebro_incident['confidence']:.0%}
Root Cause: {cerebro_incident['root_cause']}
Affected Component: {cerebro_incident['affected_component']}

Causal Chain:
""" + '\n'.join(f'  {i+1}. {s}' for i, s in enumerate(cerebro_incident['causal_chain'])) + """

Immediate Actions:
""" + '\n'.join(f'  {i+1}. {s}' for i, s in enumerate(cerebro_incident['remediation_steps']))

print('=' * 60)
print('Running Agentic Incident Response')
print('Scenario: Cerebro replication lag (only closed issue in mock store)')
print('Expected: search → find closed issue → create NEW open issue')
print('=' * 60)
print()

result = run_agentic_loop(incident_text)

print()
print('=' * 60)
print('AGENT FINAL RESPONSE')
print('=' * 60)
print(result['final_response'])

print()
print('=' * 60)
print('EXECUTION SUMMARY')
print('=' * 60)
print(f"  Total turns : {result['turns']}")
print(f"  Tool calls  : {len(result['tool_calls'])}")

print()
print('TOOL CALL DETAILS')
for i, tc in enumerate(result['tool_calls'], 1):
    print(f"  [{i}] {tc['tool']}")
    if tc['tool'] == 'create_github_issue':
        r = tc['result']
        print(f"       Created: #{r.get('number')} — {r.get('url')}")
    elif tc['tool'] == 'search_github_issues':
        count = tc['result'].get('total_count', 0)
        print(f"       Found: {count} issue(s) matching '{tc['args'].get('query')}'")
        for iss in tc['result'].get('issues', []):
            print(f"         #{iss['number']} [{iss['state']}] {iss['title'][:55]}")

Running Agentic Incident Response
Scenario: Cerebro replication lag (only closed issue in mock store)
Expected: search → find closed issue → create NEW open issue

  [Turn 1] Calling tool: search_github_issues
           Args: ['query']
           Result: {"total_count": 0, "issues": []}
  [Turn 2] Calling tool: create_github_issue
           Args: ['title', 'body', 'labels']
           Result: {"number": 1044, "url": "https://github.com/nutanix/aos/issues/1044", "state": "open"}

AGENT FINAL RESPONSE
No existing issues were found regarding Cerebro scheduler starvation during concurrent snapshot and backup jobs. I have created a new issue, [#1044](https://github.com/nutanix/aos/issues/1044), to track this incident. The operator should proceed with the remediation steps outlined in the issue, starting with verifying the current replication status via `ncli`.

EXECUTION SUMMARY
  Total turns : 3
  Tool calls  : 2

TOOL CALL DETAILS
  [1] search_github_issues
       Found: 0 issue(s) matc

In [19]:
# ── Section 10: GitHub Issue Store ───────────────────────────────────────
open_n   = sum(1 for i in github.issues if i['state'] == 'open')
closed_n = sum(1 for i in github.issues if i['state'] == 'closed')
print(f'Total: {len(github.issues)}  Open: {open_n}  Closed: {closed_n}')
print()
print(f"  {'#':>5}  {'State':6}  {'Pri':3}  {'Title'}")
print('  ' + '-' * 65)
for i in github.issues:
    title = str(i['title'])[:50]
    print(f"  #{str(i['number']):4}  {str(i['state']):6}  {str(i['priority']):3}  {title}")

new_issues = [i for i in github.issues if i['number'] > 1043]
if new_issues:
    print()
    for i in new_issues:
        print(f'  ✅ Created #{i["number"]} — {i["url"]}')
else:
    print('\n  ℹ  Agent found existing open issue — no duplicate created.')

Total: 4  Open: 3  Closed: 1

      #  State   Pri  Title
  -----------------------------------------------------------------
  #1042  open    P1   Stargate crash loop triggered by sustained disk IO
  #1038  open    P2   CVM memory OOM kills Cassandra under high metadata
  #1031  closed  P2   Cerebro replication lag exceeds RPO during peak ba
  #1044  open    P2   Cerebro replication lag causing RPO violations dur

  ✅ Created #1044 — https://github.com/nutanix/aos/issues/1044


## Section 11 - End-to-End Pipeline

Putting it all together: from raw log lines to automated GitHub issue creation.

```
┌─────────────────┐    ┌──────────────────┐    ┌──────────────────────┐
│  Log Generator  │───►│  LLM Analysis    │───►│  Agentic Response    │
│                 │    │  (Lab 6.1/6.2)   │    │  (Lab 6.3 — this)    │
│  Realistic AOS  │    │                  │    │                      │
│  log lines with │    │  root cause      │    │  GitHub search  ──►  │
│  disk IO errors │    │  severity        │    │  GitHub create  ──►  │
│  crash entries  │    │  remediation     │    │  (if no match)       │
└─────────────────┘    └──────────────────┘    └──────────┬───────────┘
                                                           │
                                                           ▼
                                               ┌──────────────────────┐
                                               │  Module 7 — FastAPI  │
                                               │  REST endpoint wraps │
                                               │  this pipeline for   │
                                               │  production use      │
                                               └──────────────────────┘
```

> **Instructor Note:** Module 7 takes this exact pipeline and wraps it in a FastAPI endpoint. A Prometheus alert or SNMP trap fires a POST request, the endpoint runs log analysis then the agentic loop — the entire process from raw alert to GitHub issue takes under 10 seconds with no human in the loop.

In [20]:
# ── Section 11: Full End-to-End Pipeline ─────────────────────────────────

def generate_disk_failure_logs(node: str = 'node-3', disk: str = 'sda3') -> list:
    """Generate realistic Nutanix AOS log lines for a disk failure scenario."""
    ts_base = '2024-11-20T14'
    return [
        f'{ts_base}:22:01.123 WARN  disk_manager  [{node}] IO latency spike on /dev/{disk}: 450ms (threshold 100ms)',
        f'{ts_base}:22:14.456 ERROR disk_manager  [{node}] Uncorrectable IO error on /dev/{disk} — sector 0x3FA2B1',
        f'{ts_base}:22:15.001 ERROR disk_manager  [{node}] Uncorrectable IO error on /dev/{disk} — sector 0x3FA2C0',
        f'{ts_base}:22:17.889 ERROR disk_manager  [{node}] Error count threshold reached (3/3) — marking /dev/{disk} FAILED',
        f'{ts_base}:22:18.002 INFO  disk_manager  [{node}] Removing /dev/{disk} from storage pool SP-NVME-01',
        f'{ts_base}:22:18.150 WARN  stargate      [{node}] Extent store reconfiguration triggered — pool SP-NVME-01 changed',
        f'{ts_base}:22:18.900 ERROR stargate      [{node}] Failed to rebuild extent map after disk removal: ENODEV',
        f'{ts_base}:22:19.001 FATAL stargate      [{node}] Unrecoverable state — initiating controlled shutdown',
        f'{ts_base}:22:19.200 INFO  genesis       [{node}] stargate exited with code 1 — scheduling restart (attempt 1/3)',
        f'{ts_base}:22:24.600 FATAL stargate      [{node}] Unrecoverable state — initiating controlled shutdown',
        f'{ts_base}:22:24.800 INFO  genesis       [{node}] stargate exited with code 1 — scheduling restart (attempt 2/3)',
        f'{ts_base}:22:29.900 FATAL stargate      [{node}] Unrecoverable state — initiating controlled shutdown',
        f'{ts_base}:22:30.100 ERROR genesis       [{node}] stargate crash loop detected — 3 failures in 12 minutes',
        f'{ts_base}:22:30.200 WARN  cluster_health         Storage pool SP-NVME-01 DEGRADED on {node} — 1 drive offline',
    ]


def quick_llm_analyse(log_lines: list) -> dict:
    """Call Gemini to extract a structured incident summary from raw logs."""
    logs_str = '\n'.join(log_lines)
    system = (
        'You are a Nutanix AOS log analyser. Return ONLY valid JSON with keys: '
        'incident_title, root_cause, affected_component, severity (P1/P2/P3), '
        'confidence (0.0-1.0), causal_chain (list of strings), '
        'remediation_steps (list of ncli commands), requires_escalation (bool).'
    )
    raw = call_gemini(logs_str, system=system, temperature=0.0)
    # Strip markdown code fences if present
    raw = raw.strip()
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1].rsplit('```', 1)[0].strip()
    return json.loads(raw)


# ── Run the full 3-step pipeline ──────────────────────────────────────────
# Reset GitHub store so the disk-failure scenario is fresh
github.issues = github.issues[:3]
github._issue_counter = 1044
executor.call_log.clear()

print('STEP 1/3 — Generating disk failure logs...')
log_lines = generate_disk_failure_logs('node-5', 'nvme0n1')
print(f'  Generated {len(log_lines)} log lines')
print(f'  First: {log_lines[0]}')
print(f'  Last:  {log_lines[-1]}')

print()
print('STEP 2/3 — LLM root-cause analysis...')
analysis = quick_llm_analyse(log_lines)
print(f"  Title    : {analysis.get('incident_title', 'N/A')}")
print(f"  Severity : {analysis.get('severity', 'N/A')}")
print(f"  Root cause: {analysis.get('root_cause', 'N/A')[:100]}")

print()
print('STEP 3/3 — Agentic incident response...')
incident_text2 = f"""Incident Title: {analysis.get('incident_title', 'Unknown')}
Severity: {analysis.get('severity', 'P2')}
Confidence: {analysis.get('confidence', 0.9):.0%}
Root Cause: {analysis.get('root_cause', '')}
Affected Component: {analysis.get('affected_component', 'stargate')}

Causal Chain:
""" + '\n'.join(f'  {i+1}. {s}' for i, s in enumerate(analysis.get('causal_chain', [])))

result2 = run_agentic_loop(incident_text2)

print()
print('PIPELINE COMPLETE')
print(f"  Turns: {result2['turns']}  |  Tools called: {len(result2['tool_calls'])}")
print()
print('Agent summary:')
print(result2['final_response'])

# Show what was created / found
new_issues2 = [i for i in github.issues if i['number'] > 1043]
if new_issues2:
    print()
    print(f'GitHub issue created: #{new_issues2[0]["number"]} — {new_issues2[0]["url"]}')
else:
    matched = result2['tool_calls'][0]['result'].get('issues', []) if result2['tool_calls'] else []
    if matched:
        print(f'Existing issue referenced: #{matched[0]["number"]} — {matched[0]["url"]}')

STEP 1/3 — Generating disk failure logs...
  Generated 14 log lines
  First: 2024-11-20T14:22:01.123 WARN  disk_manager  [node-5] IO latency spike on /dev/nvme0n1: 450ms (threshold 100ms)
  Last:  2024-11-20T14:22:30.200 WARN  cluster_health         Storage pool SP-NVME-01 DEGRADED on node-5 — 1 drive offline

STEP 2/3 — LLM root-cause analysis...
  Title    : Stargate crash loop due to NVMe drive failure and extent map corruption
  Severity : P1
  Root cause: Physical NVMe drive failure (/dev/nvme0n1) leading to unrecoverable IO errors and subsequent failure

STEP 3/3 — Agentic incident response...
  [Turn 1] Calling tool: search_github_issues
           Args: ['query']
           Result: {"total_count": 1, "issues": [{"number": 1042, "title": "Stargate crash loop triggered by sustained disk IO errors on NV

PIPELINE COMPLETE
  Turns: 2  |  Tools called: 1

Agent summary:
The search identified an existing open issue, **#1042** (https://github.com/nutanix/aos/issues/1042), which tracks

## Section 12 - MCP Best Practices

### 6 Best Practices for Production MCP Deployments

1. **Tool description quality is everything.**
   The `description` field is your primary lever for controlling LLM behaviour. Include preconditions (`"ALWAYS call this first"`), postconditions (`"Returns the issue number and URL"`), and usage guidance. Treat it as a mini system prompt for that specific tool.

2. **Right-size your tools (granularity matters).**
   One tool per atomic operation. Do not create a `handle_incident` mega-tool — create `search_github_issues` and `create_github_issue` separately. The LLM decides how to compose them; your tools provide the primitives.

3. **Always handle errors gracefully.**
   Return structured error responses (`{"error": "...", "code": 404}`) instead of raising exceptions. The LLM can reason about errors and retry with different arguments — but only if it receives a structured response, not a Python traceback.

4. **Design for idempotency.**
   Tool calls may be retried. Before `create_github_issue`, always `search_github_issues` first. The agent system prompt encodes this rule, and the search result prevents duplicate issues.

5. **Security: principle of least privilege.**
   Each MCP server should have the minimum permissions needed. The GitHub MCP server needs `Issues: read and write` — it does not need repo write access or admin permissions. Scope tokens tightly and rotate them regularly.

6. **Log every tool call with input and output.**
   The agentic loop is hard to debug without a complete audit trail. Log tool name, arguments, result, and timestamp for every call. This is also required for compliance in regulated environments (SOC2, HIPAA).

---

### Common MCP Servers

| Service | npm Package | Key Tools |
|---------|------------|----------|
| **GitHub** | `@modelcontextprotocol/server-github` | search_issues, create_issue, get_pull_request |
| **PostgreSQL** | `@modelcontextprotocol/server-postgres` | query, list_tables, describe_table |
| **Filesystem** | `@modelcontextprotocol/server-filesystem` | read_file, write_file, list_directory |
| **Fetch (HTTP)** | `@modelcontextprotocol/server-fetch` | fetch (GET/POST any URL) |
| **Slack** | `@modelcontextprotocol/server-slack` | post_message, list_channels, get_channel_history |

> **Instructor Note:** The Fetch MCP server is the Swiss Army knife for internal APIs. If your company has a Nutanix Prism Central REST API, an internal CMDB, or a custom monitoring dashboard, the Fetch MCP server lets you expose all of them to your LLM without writing any custom MCP server code. Point it at an internal API URL and the LLM can query it directly.

## Summary

| Topic | What You Learned |
|-------|------------------|
| **MCP Protocol** | Open standard (Anthropic 2024) for LLM-to-tool connections over JSON-RPC 2.0 |
| **Three Primitives** | Tools (callable functions), Resources (read-only data), Prompts (templates) |
| **Tool Schemas** | JSON Schema definitions with name, description, and inputSchema — description is prompt engineering |
| **Agentic Loop** | 4-step cycle: user message → LLM tool call → execute → function response → repeat until text |
| **Tool Descriptions** | The most important lever for controlling agent behaviour — encode business logic in descriptions |
| **Mock to Real** | Mock services have identical interfaces — swap mock classes for real GitHub API calls to go to production |

---

**Module 6 Complete.** You have built a full agentic incident response system that:
1. Receives a structured incident summary
2. Autonomously searches GitHub for known issues
3. Creates a new tracked GitHub issue if no existing match is found
4. Synthesises a final summary for the operator

All without a single line of if/else routing logic — the LLM decides the sequence based on search results.

---

**Module 7 Preview:** We wrap this pipeline in a **FastAPI** REST endpoint. A Prometheus alert or SNMP trap fires a POST request, the endpoint runs log analysis + the agentic loop, and returns the GitHub issue URL — all within 10 seconds, with no human intervention required.

## Challenges

### Challenge 1 — Real GitHub Integration
Replace `MockGitHub.search_issues()` with a real GitHub API call using `requests`.
Search the `nutanix/ntnx-api-python-client` repository (or any public repo) for open issues matching a query.
Use the GitHub REST API: `GET https://api.github.com/search/issues?q={query}+repo:{owner}/{repo}`.
No auth token needed for public repos (60 requests/hour unauthenticated).
Replace `create_issue` with a real `POST https://api.github.com/repos/{owner}/{repo}/issues` call using your GitHub token.
Verify the agentic loop still works with real results — the tool schema and executor are unchanged.

### Challenge 2 — Tool Chaining: Auto-Label by Component
Add a third tool `label_github_issue(issue_number: int, labels: list)` to the executor and tool schema.
Update `AGENT_SYSTEM` to instruct the agent: "After creating or finding an issue, call `label_github_issue` to add component labels based on the affected component field."
Verify the agent calls all three tools in sequence: search → create/find → label.
Check the mock store shows the labels added to the issue.

### Challenge 3 — Multi-Incident Batch Agent
Write a `batch_incident_response(incidents: list) -> list` function that runs `run_agentic_loop()` for each incident in the list.
Use Python's `concurrent.futures.ThreadPoolExecutor` to run up to 3 incidents in parallel.
Test with 3 different incidents: disk failure (P1), CVM memory OOM (P2), and Cerebro replication lag (P2).
Print a summary table showing incident title, tools called, GitHub issue URL, and total turns per incident.

### Challenge 4 — Duplicate Detection
Modify the AGENT_SYSTEM prompt so that if `search_github_issues` returns an OPEN issue with title similarity > 80%, the agent should comment on the existing issue body (add a note about the recurrence) instead of creating a new one.
Add a `comment_github_issue(issue_number: int, comment: str)` tool to MockGitHub and the executor.
Test by running the same Cerebro incident twice — verify the second run comments rather than creates.

In [21]:
# Challenge workspace
